# Unfallatlas 2025: Data Preparation and Splits

This notebook turns the processed accident data (`data/processed/accidents_2025.csv`: English column names plus city data)
into **train / validation / test splits** that the modelling notebooks start from.
Findings behind the choices made here are in `tiziano_eda.ipynb`; column meanings are in `data/DSB_Unfallatlas_EN.pdf`.

**Prediction task:** given the circumstances of an accident with personal injury, predict whether someone was
**killed or seriously injured** (`target = severe`, about 15.7 % positives).
The original 3-class `severity` is kept in every split, so a multi-class model can use the same files.

Workflow:
1. Load the data
2. Create stateless features
3. Split 70 / 15 / 15, stratified by `severity`
4. Save the splits to `data/splits/`

Anything that is *learned* from data (scaling, encoding, target encoding, imputation, resampling) belongs **after** the split and is fitted on the training set only.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
SPLIT_DIR = Path("data/splits")

## 1. Load

* The file is comma-separated with `.` as decimal point (reading it with `decimal=","` turns coordinates and population into strings).
* `accident_id` and the municipality key `ags` have leading zeros, so they are read as strings.
* All 273,007 rows are kept as they are. 195 rows have no city data (`city_note != "ok"`, mostly uninhabited areas)
  and 18 rows have no `date`; handle these after the split (e.g. impute on train, or drop).

In [2]:
df = pd.read_csv("data/processed/accidents_2025.csv", dtype={"accident_id": str, "ags": str})
print(df.shape)
df.head()

(273007, 41)


,accident_id,state,region,district,municipality,year,month,hour,weekday,severity,...,single_type,vulnerable_vs_vehicle,date,date_check,killed,ags,population,area_km2,population_per_km2,city_note
0,01250509171013352025,1,1-0,1-0-60,1-0-60-7,2025,5,15,Friday,Accident with slightly injured,...,1,0,2025-05-09,ok,0,01060007,1067.0,34.85,31.0,ok
1,01250405161013752025,1,1-0,1-0-62,1-0-62-53,2025,4,14,Saturday,Accident with slightly injured,...,1,0,2025-04-05,ok,0,01062053,9052.0,11.36,797.0,ok
2,01250118171013142025,1,1-0,1-0-56,1-0-56-15,2025,1,9,Saturday,Accident with slightly injured,...,1,0,2025-01-18,ok,0,01056015,51375.0,21.36,2405.0,ok
3,01250326152013912025,1,1-0,1-0-55,1-0-55-28,2025,3,16,Wednesday,Accident with slightly injured,...,0,1,2025-03-26,ok,0,01055028,10842.0,69.07,157.0,ok
4,01250225145013362025,1,1-0,1-0-58,1-0-58-116,2025,2,17,Tuesday,Accident with slightly injured,...,1,0,2025-02-25,ok,0,01058116,834.0,13.89,60.0,ok


In [3]:
df.columns

Index(['accident_id', 'state', 'region', 'district', 'municipality', 'year',
       'month', 'hour', 'weekday', 'severity', 'accident_kind',
       'accident_type', 'light', 'road_condition', 'bicycle', 'car',
       'pedestrian', 'motorcycle', 'truck', 'other_vehicle', 'utm_x', 'utm_y',
       'longitude', 'latitude', 'location_check', 'severe', 'state_name',
       'year_month', 'participants', 'n_participant_types', 'vulnerable',
       'single_type', 'vulnerable_vs_vehicle', 'date', 'date_check', 'killed',
       'ags', 'population', 'area_km2', 'population_per_km2', 'city_note'],
      dtype='str')

## 2. Stateless features

Sine/cosine encodings of hour and month, so that 23 h sits next to 0 h and December next to January.
They are computed row by row, so they are safe before the split.

In [4]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

## 3. Train / validation / test split

* **70 % train**: fit models and cross-validate for hyperparameter tuning.
* **15 % validation**: compare models and do error analysis without touching the test set.
* **15 % test**: used **once**, for the final score of the chosen model.

The split is random and stratified by `severity`. With only 0.9 % fatal accidents, stratification keeps the
same share of each severity level in every split.

`severity`, `severe` and `killed` describe the outcome, so they must never be used as features.

In [5]:
TARGET = "severe"
TARGET_MULTICLASS = "severity"
OUTCOME_COLS = ["severity", "severe", "killed"]

train, temp = train_test_split(
    df, test_size=0.30, stratify=df[TARGET_MULTICLASS], random_state=RANDOM_STATE
)
val, test = train_test_split(
    temp, test_size=0.50, stratify=temp[TARGET_MULTICLASS], random_state=RANDOM_STATE
)

splits = {"train": train, "val": val, "test": test}
summary = pd.DataFrame({
    name: {
        "rows": len(s),
        "share_%": round(len(s) / len(df) * 100, 1),
        "% killed": round(s["killed"].mean() * 100, 2),
        "% target = 1": round(s[TARGET].mean() * 100, 2),
        "rows without city data": (s["city_note"] != "ok").sum(),
    }
    for name, s in splits.items()
}).T
summary

,rows,share_%,% killed,% target = 1,rows without city data
train,191104.0,70.0,0.91,15.72,134.0
val,40951.0,15.0,0.91,15.72,36.0
test,40952.0,15.0,0.91,15.72,25.0


In [6]:
# sanity checks: no row in two splits, nothing lost
ids = {name: set(s["accident_id"]) for name, s in splits.items()}
assert df["accident_id"].is_unique
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
assert sum(len(s) for s in splits.values()) == len(df)
print("splits are disjoint and complete")

splits are disjoint and complete


## 4. Save the splits

Saved as gzip-compressed CSV (pandas reads them directly). Read the IDs back as strings.

In [7]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
for name, s in splits.items():
    s.to_csv(SPLIT_DIR / f"{name}.csv.gz", index=False)
    print(f"saved {SPLIT_DIR / f'{name}.csv.gz'}: {len(s):,} rows")

saved data/splits/train.csv.gz: 191,104 rows
saved data/splits/val.csv.gz: 40,951 rows
saved data/splits/test.csv.gz: 40,952 rows


## How to load the splits in a modelling notebook

Copy this cell. All columns are kept in the splits; pick your feature columns from `X_train`, but never the outcome columns.

In [8]:
TARGET = "severe"
OUTCOME_COLS = ["severity", "severe", "killed"]


def load_split(name):
    return pd.read_csv(
        f"data/splits/{name}.csv.gz",
        dtype={"accident_id": str, "ags": str},
        parse_dates=["date"],
    )


train = load_split("train")
val = load_split("val")

X_train, y_train = train.drop(columns=OUTCOME_COLS), train[TARGET]
X_val, y_val = val.drop(columns=OUTCOME_COLS), val[TARGET]
print(X_train.shape, X_val.shape, f"positives in train: {y_train.mean():.1%}")

(191104, 42) (40951, 42) positives in train: 15.7%


## Notes for modelling

* **Metric:** classes are imbalanced (≈ 16 % positives), so accuracy is misleading (always predicting "slight" scores 84 %).
  Use PR-AUC / average precision, recall and F1 for the positive class, and pick a decision threshold on the validation set.
* **Imbalance:** try `class_weight="balanced"` first; under- or oversampling only inside the training folds.
* **Simple baselines to beat:** a `DummyClassifier`, and a rule such as "motorcycle involved or vehicle left the carriageway"
  (both stood out as high-severity in the EDA).
* **Leakage check:** all features describe the accident itself and are known once it is recorded. Nothing describes the outcome.